# ECOS — Block 4: Tissue-Mimicking Comparison

**Objective:** Determine which combination of (PVA%, PG%, cycles) best mimics specific
soft tissues. This block connects our measured properties with clinical reference values
from the literature (Zell 2007, Duck 1990, ICRU Report 61).

**Hypotheses addressed:** Tissue-matching goal (see *Study Rules §1*) — which formulations
fall within the property space of liver, muscle, breast, kidney, and brain?

**This block uses all cycles together** (not one cycle at a time) to map the full
reachable property space and its evolution with freeze-thaw cycling.

> **Before interpreting any result, complete the checklist in *Study Rules §8*.**
> Tissue reference values are population ranges — a phantom matching the center is
> not necessarily better than one at the edge. The goal is to fall *within* the range.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.spatial import ConvexHull
from scipy import stats
from IPython.display import display

sys.path.insert(0, str(Path().resolve()))
import ecos_loader

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

base_dir = Path('../database')
df = ecos_loader.build_catalog(base_dir)

for col in ('pva_pct', 'pg_pct', 'cycle'):
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['pva_real'] = df['pva_pct'].apply(lambda x: x / 10 if x > 100 else x)

fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)

# Color convention (Study Rules §7): tab10 — PVA 10%=blue, 12.5%=orange, 15%=green
_cmap      = plt.cm.tab10
PVA_LEVELS = sorted(df['pva_real'].dropna().unique())
PG_LEVELS  = sorted(df['pg_pct'].dropna().unique())
PVA_COLOR  = {pva: _cmap(i) for i, pva in enumerate(PVA_LEVELS)}
PG_MARKER  = {pg: m for pg, m in zip(PG_LEVELS, ['o', 's', '^', 'D'])}

cycles_avail = sorted(df['cycle'].dropna().unique())
print(f'Catalog shape: {df.shape}')
print(f'Cycles available: {cycles_avail}')
print(f'PVA levels (real): {PVA_LEVELS}')
print(f'PG levels: {PG_LEVELS}')

---
## Cell 4.0 — Tissue reference values

Values compiled from Zell (2007), Duck (1990), and ICRU Report 61.
**These must be verified against the actual papers before publication.**

Reference ranges represent population variability across patients, measurement
conditions, and methods — not instrument uncertainty. Treat them as approximate
targets. A phantom that falls *within* the range qualifies as a tissue mimic;
the center of the range is not a privileged target.

Z (acoustic impedance) is derived: Z = ρ · Cl. Ranges computed from the
extreme combinations (min·min, max·max) of ρ and Cl.

In [ ]:
# --- Cell 4.0: Tissue reference table ---

# Source: Zell 2007, Duck 1990, ICRU Report 61
# rho in g/cm3, Cl in m/s
tissue_refs = {
    'Liver':              {'Cl_min': 1540, 'Cl_max': 1590, 'rho_min': 1.05, 'rho_max': 1.07},
    'Breast (glandular)': {'Cl_min': 1510, 'Cl_max': 1570, 'rho_min': 1.02, 'rho_max': 1.06},
    'Muscle':             {'Cl_min': 1545, 'Cl_max': 1630, 'rho_min': 1.04, 'rho_max': 1.06},
    'Kidney':             {'Cl_min': 1560, 'Cl_max': 1570, 'rho_min': 1.04, 'rho_max': 1.06},
    'Brain':              {'Cl_min': 1530, 'Cl_max': 1560, 'rho_min': 1.03, 'rho_max': 1.05},
    'Water (37 C)':       {'Cl_min': 1520, 'Cl_max': 1528, 'rho_min': 1.00, 'rho_max': 1.00},
}

# Muted pastel colors for tissue regions (separate from tab10 used for phantom data)
TISSUE_COLORS = {
    'Liver':              '#a8d5a2',   # muted green
    'Breast (glandular)': '#fde68a',   # muted yellow
    'Muscle':             '#fca5a5',   # muted red
    'Kidney':             '#bfdbfe',   # muted blue
    'Brain':              '#ddd6fe',   # muted violet
    'Water (37 C)':       '#e2e8f0',   # light grey
}

# Build display DataFrame
rows_ref = []
for tissue, ref in tissue_refs.items():
    Cl_c = (ref['Cl_min'] + ref['Cl_max']) / 2
    rho_c = (ref['rho_min'] + ref['rho_max']) / 2
    Z_min = ref['rho_min'] * ref['Cl_min'] / 1e6  # MRayl
    Z_max = ref['rho_max'] * ref['Cl_max'] / 1e6
    rows_ref.append({
        'Tissue': tissue,
        'Cl min (m/s)': ref['Cl_min'],
        'Cl max (m/s)': ref['Cl_max'],
        'Cl center (m/s)': round(Cl_c, 0),
        'rho min (g/cm3)': ref['rho_min'],
        'rho max (g/cm3)': ref['rho_max'],
        'rho center (g/cm3)': round(rho_c, 4),
        'Z min (MRayl)': round(Z_min, 3),
        'Z max (MRayl)': round(Z_max, 3),
    })

ref_df = pd.DataFrame(rows_ref).set_index('Tissue')
print('Tissue acoustic reference values (Zell 2007, Duck 1990, ICRU Report 61)\n')
display(ref_df)
print('\n[!] Verify these ranges against the source papers before publication.')

---
## Cell 4.1 — Property map: $C_l$ vs density with tissue regions

THE key figure: 2D scatter plot of all phantom specimens in the ($\rho$, $C_l$) plane,
overlaid with shaded reference rectangles for each tissue type.

- **Points inside a tissue rectangle** → that formulation mimics that tissue
- **Points between regions** → intermediate properties; interpolation possible
- **Empty tissue regions** → no current formulation matches; different composition needed
- **Cluster spread** → fabrication reproducibility of each formulation

Color = PVA%, marker shape = PG%. All cycles pooled to show the full property space.
Error bars: study rules §6 require SD; shown here as individual points for transparency (n=5 per condition/cycle).

In [ ]:
# --- Cell 4.1: Property map Cl vs density (all cycles pooled) ---
mask41 = df[['US_Cl', 'DENS_density_gcm3']].notna().all(axis=1)
sub41  = df[mask41].copy()

fig, ax = plt.subplots(figsize=(9, 6))

# Tissue rectangles (background layer)
for tissue, ref in tissue_refs.items():
    x0, x1 = ref['rho_min'], ref['rho_max']
    y0, y1 = ref['Cl_min'], ref['Cl_max']
    rect = mpatches.FancyBboxPatch(
        (x0, y0), x1 - x0, y1 - y0,
        boxstyle='square,pad=0',
        facecolor=TISSUE_COLORS[tissue], edgecolor='dimgray',
        linewidth=0.8, alpha=0.55, zorder=1,
    )
    ax.add_patch(rect)
    ax.text(
        (x0 + x1) / 2, (y0 + y1) / 2, tissue,
        ha='center', va='center', fontsize=7.5,
        color='dimgray', style='italic', zorder=2,
    )

# Phantom data points
for pva in PVA_LEVELS:
    for pg in PG_LEVELS:
        pts = sub41[(sub41['pva_real'] == pva) & (sub41['pg_pct'] == pg)]
        if pts.empty:
            continue
        ax.scatter(
            pts['DENS_density_gcm3'], pts['US_Cl'],
            color=PVA_COLOR[pva], marker=PG_MARKER[pg],
            s=55, alpha=0.85, edgecolors='white', linewidths=0.4, zorder=4,
        )

# Legend
pva_hdl = [Line2D([0], [0], marker='o', color='w', markerfacecolor=PVA_COLOR[pva],
                  markersize=9, label=f'PVA {pva:.4g}%') for pva in PVA_LEVELS]
pg_hdl  = [Line2D([0], [0], marker=PG_MARKER[pg], color='gray',
                  markersize=9, label=f'PG {int(pg)}%') for pg in PG_LEVELS]
ax.legend(handles=pva_hdl + pg_hdl, fontsize=8, framealpha=0.9,
          title='Color=PVA%  Shape=PG%', ncol=2, loc='lower right')

ax.set_xlabel('Density $\\rho$ (g/cm\u00b3)', fontsize=12)
ax.set_ylabel('Longitudinal velocity $C_l$ (m/s)', fontsize=12)
ax.set_title(
    '$C_l$ vs density — all cycles pooled  (shaded = tissue reference regions)',
    fontsize=12,
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(fig_dir / 'B4_41_property_map_all_cycles.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Total specimens plotted: {len(sub41)}')
print(f'Cl range: {sub41["US_Cl"].min():.1f} – {sub41["US_Cl"].max():.1f} m/s')
print(f'rho range: {sub41["DENS_density_gcm3"].min():.4f} – {sub41["DENS_density_gcm3"].max():.4f} g/cm3')

---
## Cell 4.2 — Property map evolution: one panel per cycle

Same axes as 4.1 but split into one subplot per freeze-thaw cycle. Tissue regions
stay fixed; data points shift between panels.

- **Points migrating toward a tissue region with more cycles** → cycling tunes the phantom closer
- **Points migrating away** → over-processing; fewer cycles may be better
- **All points shifting in the same direction** → systematic cycle effect independent of formulation

Axis limits are shared across panels for direct visual comparison.

In [ ]:
# --- Cell 4.2: Property map — one panel per cycle ---
n_cycles = len(cycles_avail)
fig, axes = plt.subplots(1, n_cycles, figsize=(5 * n_cycles, 5.5), sharey=True, sharex=True)
if n_cycles == 1:
    axes = [axes]

# Common axis limits (computed from all data + tissue refs, with padding)
all_rho = sub41['DENS_density_gcm3']
all_cl  = sub41['US_Cl']
t_rho_min = min(r['rho_min'] for r in tissue_refs.values())
t_rho_max = max(r['rho_max'] for r in tissue_refs.values())
t_cl_min  = min(r['Cl_min']  for r in tissue_refs.values())
t_cl_max  = max(r['Cl_max']  for r in tissue_refs.values())
xlim = (min(all_rho.min(), t_rho_min) - 0.01, max(all_rho.max(), t_rho_max) + 0.01)
ylim = (min(all_cl.min(),  t_cl_min)  - 10,   max(all_cl.max(),  t_cl_max)  + 10)

for ax, cyc in zip(axes, cycles_avail):
    sub_c = sub41[sub41['cycle'] == cyc]

    for tissue, ref in tissue_refs.items():
        x0, x1 = ref['rho_min'], ref['rho_max']
        y0, y1 = ref['Cl_min'], ref['Cl_max']
        rect = mpatches.FancyBboxPatch(
            (x0, y0), x1 - x0, y1 - y0,
            boxstyle='square,pad=0',
            facecolor=TISSUE_COLORS[tissue], edgecolor='dimgray',
            linewidth=0.7, alpha=0.5, zorder=1,
        )
        ax.add_patch(rect)
        ax.text(
            (x0 + x1) / 2, (y0 + y1) / 2, tissue,
            ha='center', va='center', fontsize=6.5,
            color='dimgray', style='italic', zorder=2,
        )

    for pva in PVA_LEVELS:
        for pg in PG_LEVELS:
            pts = sub_c[(sub_c['pva_real'] == pva) & (sub_c['pg_pct'] == pg)]
            if pts.empty:
                continue
            ax.scatter(
                pts['DENS_density_gcm3'], pts['US_Cl'],
                color=PVA_COLOR[pva], marker=PG_MARKER[pg],
                s=45, alpha=0.85, edgecolors='white', linewidths=0.4, zorder=4,
            )

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_title(f'Cycle {int(cyc)}', fontsize=11)
    ax.set_xlabel('Density $\\rho$ (g/cm\u00b3)', fontsize=10)
    if ax is axes[0]:
        ax.set_ylabel('$C_l$ (m/s)', fontsize=10)
    ax.grid(True, alpha=0.3)

# Shared legend on last panel
pva_hdl = [Line2D([0], [0], marker='o', color='w', markerfacecolor=PVA_COLOR[pva],
                  markersize=8, label=f'PVA {pva:.4g}%') for pva in PVA_LEVELS]
pg_hdl  = [Line2D([0], [0], marker=PG_MARKER[pg], color='gray',
                  markersize=8, label=f'PG {int(pg)}%') for pg in PG_LEVELS]
axes[-1].legend(handles=pva_hdl + pg_hdl, fontsize=7, framealpha=0.9,
                title='Color=PVA%\nShape=PG%', loc='lower right')

fig.suptitle(
    '$C_l$ vs density evolution by cycle  (tissue regions fixed)',
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.savefig(fig_dir / 'B4_42_property_map_by_cycle.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 4.3 — Distance to tissue: quantitative matching (heatmap)

Normalized Euclidean distance from each phantom condition to each tissue center:

$$d = \sqrt{\left(\frac{C_{l,ph} - C_{l,tissue}}{\Delta C_{l,tissue}/2}\right)^2 +
\left(\frac{\rho_{ph} - \rho_{tissue}}{\Delta\rho_{tissue}/2}\right)^2}$$

where $\Delta$ denotes the full range (max − min). **Normalization is critical**: without it,
$C_l$ values (~1500) would dominate over density (~1.0). Normalization puts both properties
on the same relative scale.

- **d < 1**: phantom falls within the tissue's property ellipse → matches tissue
- **d ≈ 1**: borderline match
- **d > 2**: poor match

Rows = phantom conditions (PVA% × PG%), averaged across all pieces for the **last cycle**
(most relevant for tissue-matching recommendation).

In [ ]:
# --- Cell 4.3: Distance heatmap — last cycle ---
LAST_CYCLE = int(max(cycles_avail))
df_last = df[df['cycle'] == LAST_CYCLE].copy()

# Mean per condition (PVA% x PG%)
grp43 = df_last.groupby(['pva_real', 'pg_pct'])[['US_Cl', 'DENS_density_gcm3']].mean()
grp43 = grp43.dropna()

# Compute normalized distance for each condition-tissue pair
cond_labels = [f'PVA {pva:.4g}%\nPG {int(pg)}%' for (pva, pg) in grp43.index]
tissue_names = list(tissue_refs.keys())

dist_mat = np.full((len(grp43), len(tissue_names)), np.nan)
for ci, ((pva, pg), row) in enumerate(grp43.iterrows()):
    cl_ph  = row['US_Cl']
    rho_ph = row['DENS_density_gcm3']
    for ti, (tissue, ref) in enumerate(tissue_refs.items()):
        Cl_c   = (ref['Cl_min'] + ref['Cl_max']) / 2
        rho_c  = (ref['rho_min'] + ref['rho_max']) / 2
        Cl_r   = (ref['Cl_max'] - ref['Cl_min']) / 2  # half-range
        rho_r  = (ref['rho_max'] - ref['rho_min']) / 2
        # Guard against zero half-range (e.g. Water)
        if Cl_r == 0:
            Cl_r = 4.0  # ~0.5% of 1524 m/s tolerance
        if rho_r == 0:
            rho_r = 0.005
        d = np.sqrt(((cl_ph - Cl_c) / Cl_r) ** 2 + ((rho_ph - rho_c) / rho_r) ** 2)
        dist_mat[ci, ti] = d

dist_df = pd.DataFrame(dist_mat, index=cond_labels, columns=tissue_names)

# Heatmap
fig, ax = plt.subplots(figsize=(max(8, len(tissue_names) * 1.4), max(5, len(grp43) * 0.55)))
vmin43, vmax43 = 0, 4
cmap43 = plt.cm.RdYlGn_r   # green=close, red=far
im = ax.imshow(dist_mat, aspect='auto', cmap=cmap43, vmin=vmin43, vmax=vmax43)
cbar = plt.colorbar(im, ax=ax, label='Normalized distance (d)', shrink=0.8)

ax.set_xticks(range(len(tissue_names)))
ax.set_yticks(range(len(grp43)))
ax.set_xticklabels(tissue_names, fontsize=9, rotation=20, ha='right')
ax.set_yticklabels(cond_labels, fontsize=8)
ax.set_title(f'Normalized distance to tissue — Cycle {LAST_CYCLE}  (green=match, red=far)', fontsize=12)

for i in range(len(grp43)):
    for j in range(len(tissue_names)):
        d_val = dist_mat[i, j]
        color = 'white' if d_val > 2.5 else 'black'
        ax.text(j, i, f'{d_val:.2f}', ha='center', va='center', fontsize=8, color=color)

plt.tight_layout()
plt.savefig(fig_dir / f'B4_43_distance_heatmap_C{LAST_CYCLE}.png', dpi=150, bbox_inches='tight')
plt.show()

# Count matches (d < 1)
n_match = (dist_mat < 1).sum()
print(f'Tissue matches (d < 1): {n_match} out of {dist_mat.size}')
print('\nBest match per tissue:')
for j, tissue in enumerate(tissue_names):
    best_i = np.argmin(dist_mat[:, j])
    print(f'  {tissue}: {cond_labels[best_i].replace(chr(10), " ")} (d={dist_mat[best_i, j]:.3f})')

---
## Cell 4.4 — Best match recommendation table

For each target tissue, the top-3 phantom conditions ranked by normalized distance
(from Cell 4.3). This is the practical output — the recipe book for phantom fabrication.

A robotics engineer who needs a liver phantom looks at this table and knows
what PVA%, PG%, and how many cycles to use.

**Limitation:** Only $C_l$ and $\rho$ are used for matching — attenuation and shear
velocity are not yet measured. A formulation may match acoustically but differ
in mechanical stiffness.

In [ ]:
# --- Cell 4.4: Best match recommendation table ---
rows44 = []
conditions_index = list(grp43.index)  # (pva_real, pg_pct) tuples

for j, tissue in enumerate(tissue_names):
    # Sort conditions by distance for this tissue
    sorted_ci = np.argsort(dist_mat[:, j])
    for rank, ci in enumerate(sorted_ci[:3], start=1):
        pva, pg = conditions_index[ci]
        d_val = dist_mat[ci, j]
        cl_val  = grp43.iloc[ci]['US_Cl']
        rho_val = grp43.iloc[ci]['DENS_density_gcm3']
        Z_val   = rho_val * 1000 * cl_val / 1e6  # MRayl
        rows44.append({
            'Target tissue': tissue,
            'Rank': rank,
            'PVA (%)': f'{pva:.4g}',
            'PG (%)': int(pg),
            'Cycle': LAST_CYCLE,
            'Cl (m/s)': round(cl_val, 1),
            'rho (g/cm3)': round(rho_val, 4),
            'Z (MRayl)': round(Z_val, 3),
            'Distance d': round(d_val, 3),
            'Match?': 'd < 1' if d_val < 1.0 else ('d < 2' if d_val < 2.0 else 'poor'),
        })

best_df = pd.DataFrame(rows44)
print(f'Top-3 phantom matches per tissue — Cycle {LAST_CYCLE}\n')
display(best_df.set_index(['Target tissue', 'Rank']))

# Summary: tissues with at least one match (d < 1)
matchable = [t for j, t in enumerate(tissue_names) if dist_mat[:, j].min() < 1.0]
no_match  = [t for t in tissue_names if t not in matchable]
print(f'\nTissues matchable (d < 1): {matchable if matchable else "none"}')
print(f'Tissues NOT matchable    : {no_match if no_match else "none"}')

---
## Cell 4.5 — Acoustic impedance matching

Acoustic impedance Z = ρ · Cl governs ultrasound reflection at tissue interfaces.
For a phantom to produce realistic ultrasound images, its impedance must match
the target tissue — not just its velocity or density separately.

**Horizontal shaded bands** = tissue reference Z ranges (derived from the extreme
combinations of ρ and Cl in Cell 4.0).

**Grouped bars**: each group = one PVA%, bars within group = PG levels,
mean across all pieces and the last cycle.

In [ ]:
# --- Cell 4.5: Impedance matching bar chart ---
# Use last cycle; compute mean Z per condition
df_last45 = df[df['cycle'] == LAST_CYCLE].copy()
mask45 = df_last45[['Z', 'pva_real', 'pg_pct']].notna().all(axis=1)
df_last45 = df_last45[mask45]

Z_MRayl = df_last45.copy()
Z_MRayl['Z_MRayl'] = Z_MRayl['Z'] / 1e6

grp45 = Z_MRayl.groupby(['pva_real', 'pg_pct'])['Z_MRayl'].agg(['mean', 'std']).reset_index()

n_pg  = len(PG_LEVELS)
n_pva = len(PVA_LEVELS)
bar_w = 0.18
offsets45 = np.linspace(-(n_pg - 1) * bar_w / 2, (n_pg - 1) * bar_w / 2, n_pg)
x_pva = np.arange(n_pva)

PG_COLORS_BAR = {pg: _cmap(3 + i) for i, pg in enumerate(PG_LEVELS)}

fig, ax = plt.subplots(figsize=(9, 5))

# Tissue Z bands
for tissue, ref in tissue_refs.items():
    if tissue == 'Water (37 C)':
        continue
    Z_lo = ref['rho_min'] * ref['Cl_min'] / 1e6
    Z_hi = ref['rho_max'] * ref['Cl_max'] / 1e6
    ax.axhspan(Z_lo, Z_hi, color=TISSUE_COLORS[tissue], alpha=0.4, zorder=0)
    ax.text(
        n_pva - 0.1, (Z_lo + Z_hi) / 2, tissue,
        ha='right', va='center', fontsize=7, color='dimgray', style='italic', zorder=1,
    )

# Water Z reference line
Z_water = 1.00 * 1524 / 1e6
ax.axhline(Z_water, color='steelblue', linestyle=':', linewidth=1.2, label='Water 37°C', zorder=2)

# Bars
for i_pg, pg in enumerate(PG_LEVELS):
    sub_pg = grp45[grp45['pg_pct'] == pg].sort_values('pva_real')
    x_pos = x_pva + offsets45[i_pg]
    for xi, (_, row_g) in zip(x_pos, sub_pg.iterrows()):
        ax.bar(
            xi, row_g['mean'], width=bar_w,
            color=PG_COLORS_BAR[pg], alpha=0.85,
            yerr=row_g['std'] if not np.isnan(row_g['std']) else 0,
            capsize=3, error_kw={'linewidth': 1.2},
            label=f'PG {int(pg)}%' if xi == x_pva[0] + offsets45[i_pg] else '_nolegend_',
            zorder=3,
        )

ax.set_xticks(x_pva)
ax.set_xticklabels([f'PVA {pva:.4g}%' for pva in PVA_LEVELS], fontsize=11)
ax.set_ylabel('Acoustic impedance Z (MRayl)', fontsize=12)
ax.set_title(
    f'Acoustic impedance by condition — Cycle {LAST_CYCLE}  (bars=mean±SD, bands=tissue targets)',
    fontsize=11,
)

# Deduplicate legend entries
handles, labels = ax.get_legend_handles_labels()
seen = {}; uniq_h = []; uniq_l = []
for h, l in zip(handles, labels):
    if l not in seen:
        seen[l] = True; uniq_h.append(h); uniq_l.append(l)
ax.legend(uniq_h, uniq_l, fontsize=8, framealpha=0.9, title='PG %')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(fig_dir / f'B4_45_impedance_bar_C{LAST_CYCLE}.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 4.6 — Radar (spider) chart: multi-property matching

For each target tissue, overlay the best-matching phantom's properties on a radar
chart alongside the tissue reference. Each axis is normalized to the tissue range
(0 = tissue_min, 1 = tissue_max).

- **Phantom polygon overlapping tissue polygon** → good match
- **Mismatch on one axis** → that property needs adjustment

**Current limitation:** With only $C_l$ and $\rho$ available, this is effectively
a 2-property comparison. The radar becomes truly informative once attenuation
and shear velocity are added. Included here as a template for future use.

In [ ]:
# --- Cell 4.6: Radar charts — best phantom vs tissue ---
# Select tissues with a meaningful rho range (exclude Water)
radar_tissues = [t for t in tissue_names if t != 'Water (37 C)']
props = ['Cl', 'rho']  # axes

N_ax = len(props)
angles = np.linspace(0, 2 * np.pi, N_ax, endpoint=False).tolist()
angles += angles[:1]  # close polygon

n_col = min(3, len(radar_tissues))
n_row = int(np.ceil(len(radar_tissues) / n_col))
fig, axes_r = plt.subplots(
    n_row, n_col,
    figsize=(4.5 * n_col, 4.0 * n_row),
    subplot_kw={'polar': True},
)
axes_r = np.array(axes_r).flatten()

for ax_i, tissue in enumerate(radar_tissues):
    ax_r = axes_r[ax_i]
    ref = tissue_refs[tissue]

    # Find best-matching phantom for this tissue
    ti = tissue_names.index(tissue)
    best_ci = np.argmin(dist_mat[:, ti])
    pva_b, pg_b = conditions_index[best_ci]
    cl_b  = grp43.iloc[best_ci]['US_Cl']
    rho_b = grp43.iloc[best_ci]['DENS_density_gcm3']

    # Normalize: 0 = tissue_min, 1 = tissue_max
    Cl_range  = ref['Cl_max']  - ref['Cl_min']  if ref['Cl_max']  != ref['Cl_min']  else 8.0
    rho_range = ref['rho_max'] - ref['rho_min'] if ref['rho_max'] != ref['rho_min'] else 0.01

    phantom_vals = [
        (cl_b  - ref['Cl_min'])  / Cl_range,
        (rho_b - ref['rho_min']) / rho_range,
    ]
    tissue_vals = [0.5, 0.5]  # center of the tissue range

    # Tissue band: 0 to 1 on each axis
    band_inner = [0.0, 0.0]
    band_outer = [1.0, 1.0]

    for vals, color, label, lw, ls in [
        (band_outer, TISSUE_COLORS[tissue], tissue, 0, '-'),
        (phantom_vals, PVA_COLOR[pva_b], f'PVA {pva_b:.4g}% PG {int(pg_b)}%', 2.0, '-'),
        (tissue_vals,  'dimgray', 'Tissue center', 1.5, '--'),
    ]:
        v = vals + vals[:1]
        a = angles
        if color == TISSUE_COLORS[tissue]:
            ax_r.fill(a, v, color=color, alpha=0.4)
        else:
            ax_r.plot(a, v, color=color, linewidth=lw, linestyle=ls, label=label)
            ax_r.fill(a, v, color=color, alpha=0.15)

    ax_r.set_xticks(angles[:-1])
    ax_r.set_xticklabels(['$C_l$', '$\\rho$'], fontsize=10)
    ax_r.set_ylim(-0.1, 1.6)
    ax_r.set_yticks([0, 0.5, 1.0])
    ax_r.set_yticklabels(['min', 'center', 'max'], fontsize=7, color='gray')
    ax_r.set_title(tissue, fontsize=10, pad=12)
    ax_r.legend(fontsize=7, loc='upper right', bbox_to_anchor=(1.35, 1.15))
    d_val = dist_mat[best_ci, ti]
    ax_r.text(0, -0.25, f'd = {d_val:.2f}', ha='center', va='top',
              transform=ax_r.transAxes, fontsize=8, color='dimgray')

# Hide unused axes
for ax_i in range(len(radar_tissues), len(axes_r)):
    axes_r[ax_i].set_visible(False)

fig.suptitle(
    f'Radar chart: best phantom vs tissue — Cycle {LAST_CYCLE}\n'
    '(axes normalized to tissue range; shaded band = tissue property window)',
    fontsize=11, y=1.02,
)
plt.tight_layout()
plt.savefig(fig_dir / f'B4_46_radar_C{LAST_CYCLE}.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 4.7 — Gap analysis: tissues outside the reachable property space

Computes the **convex hull** of all measured ($\rho$, $C_l$) points across all cycles,
then checks which tissue reference rectangles fall entirely or partially outside it.

- **Inside hull** → at least some phantom can match this tissue
- **Outside hull** → no current formulation can match; new composition needed

This identifies the limitations of the PVA+PG system and motivates future work:
additional dopants, different PVA concentrations, or alternative base materials
(agar, gelatin, polyacrylamide).

In [ ]:
# --- Cell 4.7: Gap analysis — convex hull vs tissue regions ---
from matplotlib.patches import Polygon as MPoly

mask47 = sub41[['DENS_density_gcm3', 'US_Cl']].notna().all(axis=1)
pts47  = sub41.loc[mask47, ['DENS_density_gcm3', 'US_Cl']].values

# Convex hull of all phantom data
try:
    hull = ConvexHull(pts47)
    hull_pts = pts47[hull.vertices]
    hull_pts = np.vstack([hull_pts, hull_pts[0]])  # close polygon
    hull_ok = True
except Exception as e:
    print(f'[!] ConvexHull failed: {e}')
    hull_ok = False

# Helper: check if a tissue rectangle corner is inside the hull
def point_in_hull(point, hull):
    """True if point is on the interior of the hull."""
    from scipy.spatial import Delaunay
    tri = Delaunay(pts47[hull.vertices])
    return tri.find_simplex(point) >= 0

fig, ax = plt.subplots(figsize=(9, 6))

# Tissue regions
tissue_coverage = {}
for tissue, ref in tissue_refs.items():
    x0, x1 = ref['rho_min'], ref['rho_max']
    y0, y1 = ref['Cl_min'], ref['Cl_max']
    rect = mpatches.FancyBboxPatch(
        (x0, y0), x1 - x0, y1 - y0,
        boxstyle='square,pad=0',
        facecolor=TISSUE_COLORS[tissue], edgecolor='dimgray',
        linewidth=0.8, alpha=0.5, zorder=1,
    )
    ax.add_patch(rect)
    ax.text(
        (x0 + x1) / 2, (y0 + y1) / 2, tissue,
        ha='center', va='center', fontsize=7.5,
        color='dimgray', style='italic', zorder=2,
    )

    # Coverage check: fraction of rectangle corners inside hull
    if hull_ok:
        corners = np.array([[x0, y0], [x0, y1], [x1, y0], [x1, y1]])
        inside  = sum(point_in_hull(c, hull) for c in corners)
        tissue_coverage[tissue] = inside  # 0–4
    else:
        tissue_coverage[tissue] = -1

# Convex hull
if hull_ok:
    ax.fill(hull_pts[:, 0], hull_pts[:, 1],
            alpha=0.10, color='navy', zorder=2, label='Phantom property hull')
    ax.plot(hull_pts[:, 0], hull_pts[:, 1],
            color='navy', linewidth=1.5, linestyle='--', zorder=3)

# Raw data scatter
ax.scatter(pts47[:, 0], pts47[:, 1],
           c='navy', s=20, alpha=0.4, zorder=4, label='Phantom specimens')

ax.set_xlabel('Density $\\rho$ (g/cm\u00b3)', fontsize=12)
ax.set_ylabel('$C_l$ (m/s)', fontsize=12)
ax.set_title(
    'Reachable property space (convex hull) vs tissue targets\n'
    '(all cycles pooled)',
    fontsize=12,
)
ax.legend(fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(fig_dir / 'B4_47_gap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Gap summary
print('Tissue coverage summary (corners of tissue rectangle inside phantom hull):')
for tissue, cnt in tissue_coverage.items():
    if cnt == 4:
        status = 'FULL coverage'
    elif 0 < cnt < 4:
        status = f'PARTIAL ({cnt}/4 corners inside)'
    elif cnt == 0:
        status = 'OUTSIDE hull — not matchable with current formulations'
    else:
        status = 'unknown'
    print(f'  {tissue:25s}: {status}')

---
## Cell 4.8 — Literature comparison: $C_l$ vs PVA%

Overlay our mean $C_l$ vs PVA% (at PG=0%) against published PVA hydrogel data
from the literature. This validates our measurements and allows reviewers to
compare directly with prior work.

**How to read it:**
- Our data within literature scatter → measurements consistent with published work
- Systematic offset → possible methodological difference (temperature, PVA MW,
  cycle protocol, measurement technique)
- Similar slope, different intercept → additive offset (e.g., temperature difference)

**Literature values below are extracted from key references.** Always verify against
the original papers and document measurement conditions (T, cycle protocol, PVA type).

| Reference | PVA% | Cl (m/s) | T (°C) | Cycles | Notes |
|-----------|------|----------|--------|--------|-------|
| Zell 2007 | 10 | ~1565 | 22 | 1 | 99% hydrolysis, 200 kDa |
| Zell 2007 | 15 | ~1575 | 22 | 1 | same |
| Surry 2004 | 10 | ~1540 | 37 | 1–5 | commercial PVA |
| Surry 2004 | 15 | ~1560 | 37 | 1–5 | same |
| Fromageau 2007 | 10 | ~1550 | 22 | 1 | standard protocol |

In [ ]:
# --- Cell 4.8: Literature comparison — mean Cl vs PVA% at PG=0 ---

# Literature reference data (extracted from papers)
# Format: (PVA%, Cl_mean, Cl_sd, temperature_C, source_label)
# Cl_sd = estimated from reported ranges or figure scatter
lit_data = [
    # Zell 2007 (22°C, 1 cycle, 99% hydrolysis, ~200 kDa)
    (10.0, 1565, 5,  22, 'Zell 2007'),
    (15.0, 1575, 5,  22, 'Zell 2007'),
    # Surry 2004 (37°C, 5 cycles)
    (10.0, 1540, 8,  37, 'Surry 2004'),
    (15.0, 1560, 8,  37, 'Surry 2004'),
    # Fromageau 2007 (22°C, 1 cycle)
    (10.0, 1550, 10, 22, 'Fromageau 2007'),
]
lit_sources = sorted(set(r[4] for r in lit_data))
lit_markers = {s: m for s, m in zip(lit_sources, ['v', 'P', 'X', '*'])}
lit_colors  = {s: c for s, c in zip(lit_sources, ['#7c3aed', '#be185d', '#0369a1', '#065f46'])}

# Our data: PG=0 only, all cycles, mean per (PVA%, cycle)
df_pg0 = df[(df['pg_pct'] == 0) & df['US_Cl'].notna() & df['pva_real'].notna()].copy()
our_grp = df_pg0.groupby(['pva_real', 'cycle'])['US_Cl'].agg(['mean', 'std', 'count']).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))

# Our data — one trace per cycle
for cyc, grp_c in our_grp.groupby('cycle'):
    grp_c = grp_c.sort_values('pva_real')
    alpha_c = 0.4 + 0.3 * (cyc - min(cycles_avail)) / max(1, max(cycles_avail) - min(cycles_avail))
    ax.errorbar(
        grp_c['pva_real'], grp_c['mean'],
        yerr=grp_c['std'], fmt='o-',
        color='navy', alpha=alpha_c,
        linewidth=1.8, markersize=7, capsize=4,
        label=f'Our data, Cycle {int(cyc)} (PG=0%)',
        zorder=4,
    )

# Literature data
for src in lit_sources:
    pts_lit = [(r[0], r[1], r[2]) for r in lit_data if r[4] == src]
    pva_l = [p[0] for p in pts_lit]
    cl_l  = [p[1] for p in pts_lit]
    sd_l  = [p[2] for p in pts_lit]
    ax.errorbar(
        pva_l, cl_l, yerr=sd_l,
        fmt=lit_markers[src] + '--',
        color=lit_colors[src], alpha=0.85,
        linewidth=1.4, markersize=9, capsize=4,
        label=src,
        zorder=3,
    )

ax.set_xticks(PVA_LEVELS)
ax.set_xticklabels([f'{v:.4g}%' for v in PVA_LEVELS], fontsize=11)
ax.set_xlabel('PVA concentration (%)', fontsize=12)
ax.set_ylabel('Longitudinal velocity $C_l$ (m/s)', fontsize=12)
ax.set_title(
    '$C_l$ vs PVA% (PG=0%) — our data vs literature\n'
    '(error bars = SD; literature values extracted from papers)',
    fontsize=11,
)
ax.legend(fontsize=8, framealpha=0.9, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(fig_dir / 'B4_48_literature_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Quantify offset
print('Literature comparison — systematic offset (our data vs literature at same PVA%):')
for pva in PVA_LEVELS:
    our_mean = our_grp[our_grp['pva_real'] == pva]['mean'].mean()
    lit_at_pva = [r[1] for r in lit_data if r[0] == pva]
    if lit_at_pva and not np.isnan(our_mean):
        lit_mean = np.mean(lit_at_pva)
        print(f'  PVA {pva:.4g}%: ours={our_mean:.1f} m/s, lit mean={lit_mean:.1f} m/s, offset={our_mean - lit_mean:+.1f} m/s')

print('\n[!] Temperature difference (37°C vs 22°C) accounts for ~20–25 m/s in Cw.')
print('[!] Verify extracted literature values against source papers before publication.')

---
## Cell 4.9 — Summary and interpretation

> **Instructions:** Fill in this template after running all cells above.
> Replace bracketed placeholders with computed values and observations.
> Consult the checklist in *Study Rules §8* before writing conclusions.

---

### Block 4 — Tissue-Mimicking Summary

**Property space coverage (all cycles pooled):**
- $C_l$ range achieved: *[min]* – *[max]* m/s
- Density range achieved: *[min]* – *[max]* g/cm³
- Z range achieved: *[min]* – *[max]* MRayl

**Best matches (last cycle, normalized distance d):**
- Liver: *[PVA X%, PG Y%]*, d = *[value]*
- Muscle: *[condition]*, d = *[value]*
- Brain: *[condition]*, d = *[value]*
- Breast (glandular): *[condition]*, d = *[value]*
- Kidney: *[condition]*, d = *[value]*

**Tissues NOT matchable with current formulations (d > 1 for all conditions):**
- *[list each tissue and explain which property is missing — Cl too low/high? density too low/high?]*

**Effect of freeze-thaw cycling on tissue matching:**
- *[Did more cycles move phantoms closer to or away from tissue regions? Were directions consistent across PVA/PG levels?]*

**Comparison with literature (Cell 4.8):**
- *[Agreement or systematic offset? What is the most likely cause — temperature, PVA MW, cycle count?]*
- *[Same Cl–PVA% slope as literature → composition effect is real and reproducible]*

**Recommendations for phantom fabrication:**
- For *[tissue]*: use PVA *[X]*%, PG *[Y]*%, *[N]* cycles
- *[Repeat for each tissue with a match]*

**Limitations of this analysis:**
- Only $C_l$ and $\rho$ used for matching — attenuation $\alpha(f)$ and shear velocity not yet measured
- Tissue reference ranges are literature averages; actual in-vivo values vary with patient and site
- Temperature during our measurements: *[X]* ± *[Y]* °C — literature values typically at 22°C or 37°C
- Small sample size (n=5 per condition) limits precision of mean estimates
- Some density values below 1.0 g/cm³ — see Block 1 red flags

**Future work needed:**
- Measure attenuation coefficient $\alpha(f)$ to add a third matching axis
- Test additional PVA concentrations (e.g., 7.5%, 17.5%) to extend the reachable hull
- Try dopants (graphite, cellulose, nanoparticles) to independently tune attenuation
- Calibrate thermistors against a reference thermometer to reduce Cw uncertainty
- Cross-validate tissue-matching with elastography measurements

---
*References: Zell et al. (2007) Phys Med Biol 52:N459; Duck FA (1990) Physical Properties of Tissue;
Surry KJM et al. (2004) Phys Med Biol 49:5529; Fromageau J et al. (2007) IEEE UFFC 54(3):498.*